# import Libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
%matplotlib inline

# Read Data

In [2]:
charging= pd.read_csv("data/cleaned_chargingdata.csv")
weather=pd.read_csv("data/cleaned_weather_data.csv")

In [3]:
# I don't know why the connectionTime data from the original dataset cannot be directly converted into the "hour:minute:second" format. 
# so I convert this column into local time again.
charging['connectionTime'] = pd.to_datetime(charging['connectionTime'],utc=True)
charging['connectionTime']=charging['connectionTime'].dt.tz_convert('America/Los_Angeles')
charging['connectionTime']= pd.to_datetime(charging['connectionTime']).dt.strftime('%Y-%m-%d %H:%M:%S')
weather['timestamp'] = pd.to_datetime(weather['timestamp'],utc=True)
weather['timestamp']=weather['timestamp'].dt.tz_convert('America/Los_Angeles')
weather['timestamp']= pd.to_datetime(weather['timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')

In [4]:
max_time_weather = weather['timestamp'].max()
min_time_weather = weather['timestamp'].min()
print(max_time_weather)
print(min_time_weather)
# the weather data start form 2018-04-25 03:42:00-07:00 to 2020-12-31 23:53:00-08:00

2020-12-31 23:53:00
2018-04-25 03:42:00


In [5]:
max_time_charging = charging['connectionTime'].max()
min_time_charging = charging['connectionTime'].min()
print(max_time_charging)
print(min_time_charging)
# the charging data start form 2018-04-25 04:08:04-07:00 to 2021-09-13 22:43:39-07:00

2021-09-13 22:43:39
2018-04-25 04:08:04


In [6]:
# There is missing weather data from 2020-12-31 23:53:00 to 2021-09-13 15:43:39
# I used the weather data from the same period of the previous year to fill in the missing data.
weather['timestamp'] = pd.to_datetime(weather['timestamp'])
weather.set_index('timestamp', inplace=True)
start_date = pd.to_datetime('2019-12-31 23:53:00')
end_date = pd.to_datetime('2020-09-13 22:43:39')
reference_time=weather.loc[start_date:end_date]
reference_time.index=reference_time.index+pd.DateOffset(years=1)
reference_time.head(25)

,Unnamed: 0,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
timestamp,,,,,,,,
2020-12-31 23:53:00,16612,9.0,33.0,Fair,989.77,0.0,0.0,9.0
2021-01-01 00:53:00,16613,9.0,33.0,Fair,989.77,11.0,0.0,7.0
2021-01-01 01:53:00,16614,11.0,33.0,Fair,990.10,6.0,0.0,11.0
2021-01-01 02:53:00,16615,9.0,29.0,Partly Cloudy,990.10,0.0,0.0,9.0
2021-01-01 03:53:00,16616,8.0,29.0,Partly Cloudy,989.11,0.0,0.0,8.0
2021-01-01 04:53:00,16617,7.0,29.0,Partly Cloudy,989.77,0.0,0.0,7.0
2021-01-01 05:53:00,16618,7.0,33.0,Fair,989.44,0.0,0.0,7.0
2021-01-01 06:53:00,16619,7.0,34.0,Fair,989.44,6.0,0.0,7.0
2021-01-01 07:53:00,16620,8.0,34.0,Fair,989.77,7.0,0.0,6.0


In [7]:
weather_full=pd.concat([weather,reference_time])
weather_full.index

DatetimeIndex(['2018-04-25 03:42:00', '2018-04-25 03:53:00',
               '2018-04-25 04:53:00', '2018-04-25 05:10:00',
               '2018-04-25 05:53:00', '2018-04-25 06:24:00',
               '2018-04-25 06:53:00', '2018-04-25 07:53:00',
               '2018-04-25 08:05:00', '2018-04-25 08:53:00',
               ...
               '2021-09-13 13:02:00', '2021-09-13 13:53:00',
               '2021-09-13 14:53:00', '2021-09-13 15:53:00',
               '2021-09-13 16:53:00', '2021-09-13 17:53:00',
               '2021-09-13 18:53:00', '2021-09-13 19:53:00',
               '2021-09-13 20:53:00', '2021-09-13 21:53:00'],
              dtype='datetime64[ns]', name='timestamp', length=33036, freq=None)

In [10]:
charging['time']=charging['connectionTime']
charging['time']=pd.to_datetime(charging['time'])
charging=charging.set_index('time')
charging.sort_index (inplace=True)
print(charging.index)

DatetimeIndex(['2018-04-25 04:08:04', '2018-04-25 06:45:10',
               '2018-04-25 06:45:50', '2018-04-25 07:37:06',
               '2018-04-25 07:40:34', '2018-04-25 07:43:50',
               '2018-04-25 07:47:42', '2018-04-25 07:58:25',
               '2018-04-25 08:10:52', '2018-04-25 08:12:11',
               ...
               '2021-09-13 12:53:30', '2021-09-13 13:10:59',
               '2021-09-13 13:51:51', '2021-09-13 14:12:53',
               '2021-09-13 14:17:04', '2021-09-13 14:17:04',
               '2021-09-13 14:37:59', '2021-09-13 16:11:12',
               '2021-09-13 18:08:16', '2021-09-13 22:43:39'],
              dtype='datetime64[ns]', name='time', length=75329, freq=None)


In [11]:
#Merge weather and charging data
# merge_asof function used to merge two datasets based on the nearest matching values, typically for time-series data or ordered data.
charging = charging.sort_values(by='time')
weather_full = weather_full.sort_values(by='timestamp')
processed_data_all= pd.merge_asof(charging.reset_index(), weather_full.reset_index(), left_on='time', right_on='timestamp', direction='nearest')
print(processed_data_all)

                     time  Unnamed: 0.1  Unnamed: 0_x  \
0     2018-04-25 04:08:04         38209             0   
1     2018-04-25 06:45:10         38210             1   
2     2018-04-25 06:45:50         38211             2   
3     2018-04-25 07:37:06         38212             3   
4     2018-04-25 07:40:34         38213             4   
...                   ...           ...           ...   
75324 2021-09-13 14:17:04         58109          3030   
75325 2021-09-13 14:37:59         25628          5873   
75326 2021-09-13 16:11:12         58111          3032   
75327 2021-09-13 18:08:16         25629          5874   
75328 2021-09-13 22:43:39         25630          5875   

                             id       connectionTime  \
0      5bc90cb9f9af8b0d7fe77cd2  2018-04-25 04:08:04   
1      5bc90cb9f9af8b0d7fe77cd3  2018-04-25 06:45:10   
2      5bc90cb9f9af8b0d7fe77cd4  2018-04-25 06:45:50   
3      5bc90cb9f9af8b0d7fe77cd5  2018-04-25 07:37:06   
4      5bc90cb9f9af8b0d7fe77cd6  20

In [12]:
processed_data_all=processed_data_all.set_index('time')

In [13]:
processed_data_all.head(25)

,Unnamed: 0.1,Unnamed: 0_x,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,...,requestedDeparture,timestamp,Unnamed: 0_y,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation,felt_temperature
time,,,,,,,,,,,,,,,,,,,,,
2018-04-25 04:08:04,38209,0,5bc90cb9f9af8b0d7fe77cd2,2018-04-25 04:08:04,2018-04-25 06:20:10-07:00,2018-04-25 06:21:10-07:00,7.932,2_39_78_362_2018-04-25 11:08:04.400812,2,CA-496,...,NaN,2018-04-25 03:53:00,1,12.0,26.0,Cloudy,989.11,11.0,0.0,12.0
2018-04-25 06:45:10,38210,1,5bc90cb9f9af8b0d7fe77cd3,2018-04-25 06:45:10,2018-04-25 17:56:16-07:00,2018-04-25 09:44:15-07:00,10.013,2_39_95_27_2018-04-25 13:45:09.617470,2,CA-319,...,NaN,2018-04-25 06:53:00,6,12.0,20.0,Fog,989.44,7.0,0.0,12.0
2018-04-25 06:45:50,38211,2,5bc90cb9f9af8b0d7fe77cd4,2018-04-25 06:45:50,2018-04-25 16:04:45-07:00,2018-04-25 07:51:44-07:00,5.257,2_39_79_380_2018-04-25 13:45:49.962001,2,CA-489,...,NaN,2018-04-25 06:53:00,6,12.0,20.0,Fog,989.44,7.0,0.0,12.0
2018-04-25 07:37:06,38212,3,5bc90cb9f9af8b0d7fe77cd5,2018-04-25 07:37:06,2018-04-25 16:55:34-07:00,2018-04-25 09:05:22-07:00,5.177,2_39_79_379_2018-04-25 14:37:06.460772,2,CA-327,...,NaN,2018-04-25 07:53:00,7,12.0,26.0,Cloudy,990.10,6.0,0.0,12.0
2018-04-25 07:40:34,38213,4,5bc90cb9f9af8b0d7fe77cd6,2018-04-25 07:40:34,2018-04-25 16:03:12-07:00,2018-04-25 10:40:30-07:00,10.119,2_39_79_381_2018-04-25 14:40:33.638896,2,CA-490,...,NaN,2018-04-25 07:53:00,7,12.0,26.0,Cloudy,990.10,6.0,0.0,12.0
2018-04-25 07:43:50,38214,5,5bc90cb9f9af8b0d7fe77cd7,2018-04-25 07:43:50,2018-04-25 18:17:30-07:00,2018-04-25 09:18:28-07:00,7.910,2_39_139_28_2018-04-25 14:43:49.647430,2,CA-303,...,NaN,2018-04-25 07:53:00,7,12.0,26.0,Cloudy,990.10,6.0,0.0,12.0
2018-04-25 07:47:42,38215,6,5bc90cb9f9af8b0d7fe77cd8,2018-04-25 07:47:42,2018-04-25 11:27:51-07:00,2018-04-25 11:27:42-07:00,15.294,2_39_91_441_2018-04-25 14:47:41.776352,2,CA-499,...,NaN,2018-04-25 07:53:00,7,12.0,26.0,Cloudy,990.10,6.0,0.0,12.0
2018-04-25 07:58:25,38216,7,5bc90cb9f9af8b0d7fe77cd9,2018-04-25 07:58:25,2018-04-25 12:06:29-07:00,2018-04-25 09:48:29-07:00,6.953,2_39_79_377_2018-04-25 14:58:25.255583,2,CA-325,...,NaN,2018-04-25 07:53:00,7,12.0,26.0,Cloudy,990.10,6.0,0.0,12.0
2018-04-25 08:10:52,38217,8,5bc90cb9f9af8b0d7fe77cda,2018-04-25 08:10:52,2018-04-25 11:15:46-07:00,2018-04-25 09:07:56-07:00,2.174,2_39_79_382_2018-04-25 15:10:51.871020,2,CA-491,...,NaN,2018-04-25 08:05:00,8,13.0,21.0,Haze,990.10,6.0,0.0,13.0


In [14]:
processed_data_all.to_csv('processed_data_all.csv', index=False)